# 01: Exploring the data

Where each finding in `DECISIONS.md` (Phase 1) comes from. Everything here uses the 2024-25 regular season.

In [1]:
import time

from nba_api.stats.endpoints import boxscoretraditionalv3, leaguegamelog, scoreboardv3

SEASON = "2024-25"
PAUSE = 1.5  # NBA.com times out if calls come too fast

## Load one season

One `LeagueGameLog` call returns every player's line for the whole season, and another returns every team's.

In [2]:
start = time.monotonic()
players_df = leaguegamelog.LeagueGameLog(
    season=SEASON,
    player_or_team_abbreviation="P",
    season_type_all_star="Regular Season",
    timeout=60,
).get_data_frames()[0]
print(f"players: {len(players_df):,} rows in {time.monotonic() - start:.1f}s")

time.sleep(PAUSE)
teams_df = leaguegamelog.LeagueGameLog(
    season=SEASON,
    player_or_team_abbreviation="T",
    season_type_all_star="Regular Season",
    timeout=60,
).get_data_frames()[0]
print(f"teams: {len(teams_df):,} rows")
print(f"games: {players_df['GAME_ID'].nunique():,}, players: {players_df['PLAYER_ID'].nunique():,}")

players: 26,306 rows in 0.4s


teams: 2,460 rows
games: 1,230, players: 569


In [3]:
players_df.columns.tolist()

['SEASON_ID',
 'PLAYER_ID',
 'PLAYER_NAME',
 'TEAM_ID',
 'TEAM_ABBREVIATION',
 'TEAM_NAME',
 'GAME_ID',
 'GAME_DATE',
 'MATCHUP',
 'WL',
 'MIN',
 'FGM',
 'FGA',
 'FG_PCT',
 'FG3M',
 'FG3A',
 'FG3_PCT',
 'FTM',
 'FTA',
 'FT_PCT',
 'OREB',
 'DREB',
 'REB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'PLUS_MINUS',
 'FANTASY_PTS',
 'VIDEO_AVAILABLE']

## IDs and seasons

`GAME_ID` has leading zeros, so it stays text. The first digit of `SEASON_ID` is the season type.

In [4]:
print(repr(players_df["GAME_ID"].iloc[0]), players_df["GAME_ID"].dtype)
print(players_df["SEASON_ID"].unique())

'0022400061' str
<StringArray>
['22024']
Length: 1, dtype: str


## Do player points add up to the team score?

In [5]:
player_pts = players_df.groupby(["GAME_ID", "TEAM_ID"])["PTS"].sum().rename("player_pts")
check = teams_df.set_index(["GAME_ID", "TEAM_ID"])[["PTS"]].join(player_pts)
mismatches = check[check["PTS"] != check["player_pts"]]
print(f"{len(mismatches)} mismatches out of {len(check):,} team games")

0 mismatches out of 2,460 team games


## Home and away from `MATCHUP`

Every game should have exactly one home team (`vs.`). Some don't.

In [6]:
teams_df["is_home"] = teams_df["MATCHUP"].str.contains(" vs. ")
home_count = teams_df.groupby("GAME_ID")["is_home"].sum()
print(home_count.value_counts())

no_home = home_count[home_count == 0].index
teams_df[teams_df["GAME_ID"].isin(no_home)][["GAME_ID", "GAME_DATE", "MATCHUP"]].sort_values("GAME_ID")

is_home
1    1225
0       5
Name: count, dtype: int64


,GAME_ID,GAME_DATE,MATCHUP
162,0022400147,2024-11-02,MIA @ WAS
178,0022400147,2024-11-02,WAS @ MIA
1295,0022400621,2025-01-23,IND @ SAS
1304,0022400621,2025-01-23,SAS @ IND
1323,0022400633,2025-01-25,IND @ SAS
1335,0022400633,2025-01-25,SAS @ IND
749,0022401229,2024-12-14,MIL @ ATL
750,0022401229,2024-12-14,ATL @ MIL
748,0022401230,2024-12-14,HOU @ OKC
751,0022401230,2024-12-14,OKC @ HOU


These are neutral-site games: Mexico City (Nov 2), the NBA Cup semifinals in Las Vegas (Dec 14) and Paris (Jan 23 and 25). Home and away have to come from the scoreboard instead.

## Minutes

Team minutes show overtimes (240 = regulation, +25 per overtime). `LeagueGameLog` rounds player minutes to whole numbers.

In [7]:
print(teams_df["MIN"].value_counts())
print(players_df["MIN"].dtype, players_df["MIN"].head(3).tolist())

MIN
240    2340
265     112
290       8
Name: count, dtype: int64
int64 [26, 31, 35]


Rows with 0 minutes aren't the same as not playing: some of them have stats.

In [8]:
zero = players_df[players_df["MIN"] == 0]
has_stats = (zero[["PTS", "REB", "AST", "FGA"]].sum(axis=1) > 0).sum()
print(f"{len(zero)} rows with 0 minutes, {has_stats} of them with stats")

100 rows with 0 minutes, 15 of them with stats


## Columns I won't store

In [9]:
print("empty 3P%:", players_df["FG3_PCT"].isna().sum())
[c for c in players_df.columns if c in ("FANTASY_PTS", "VIDEO_AVAILABLE")]

empty 3P%: 5362


['FANTASY_PTS', 'VIDEO_AVAILABLE']

## Players on more than one team

Trades mean a player's team can change mid-season, so the team goes on each game row.

In [10]:
teams_per_player = players_df.groupby("PLAYER_ID")["TEAM_ID"].nunique()
print(f"{(teams_per_player > 1).sum()} players played for more than one team")

81 players played for more than one team


## The nightly endpoints

`ScoreboardV3` lists a day's games. `nba_api` warns that `ScoreboardV2` has broken line scores for early 2025-26 games, so I use V3.

In [11]:
time.sleep(PAUSE)
scoreboard = scoreboardv3.ScoreboardV3(game_date="2025-03-15", timeout=60)
[(i, df.shape) for i, df in enumerate(scoreboard.get_data_frames())]

[(0, (1, 3)),
 (1, (8, 18)),
 (2, (16, 12)),
 (3, (16, 12)),
 (4, (16, 13)),
 (5, (40, 6))]

In [12]:
games = scoreboard.get_data_frames()[1]
games.columns.tolist()

['gameId',
 'gameCode',
 'gameStatus',
 'gameStatusText',
 'period',
 'gameClock',
 'gameTimeUTC',
 'gameEt',
 'regulationPeriods',
 'seriesGameNumber',
 'gameLabel',
 'gameSubLabel',
 'seriesText',
 'ifNecessary',
 'seriesConference',
 'poRoundDesc',
 'gameSubtype',
 'isNeutral']

`gameCode` is the date, then the away team, then the home team (`BOSBKN` = Boston at Brooklyn). `isNeutral` marks neutral-site games, like the NBA Cup semifinals.

In [13]:
games[["gameId", "gameCode", "isNeutral"]].head(3)

,gameId,gameCode,isNeutral
0,0022400968,20250315/BOSBKN,False
1,0022400969,20250315/OKCDET,False
2,0022400970,20250315/CHIHOU,False


In [14]:
time.sleep(PAUSE)
cup = scoreboardv3.ScoreboardV3(game_date="2024-12-14", timeout=60).get_data_frames()[1]
cup[["gameId", "gameCode", "isNeutral", "gameLabel"]]

,gameId,gameCode,isNeutral,gameLabel
0,0022401229,20241214/ATLMIL,True,Emirates NBA Cup
1,0022401230,20241214/HOUOKC,True,Emirates NBA Cup


The box score for one game uses different column names from `LeagueGameLog`, and minutes as `MM:SS`.

In [15]:
game_id = players_df[players_df["GAME_DATE"] == "2025-03-15"]["GAME_ID"].iloc[0]
time.sleep(PAUSE)
box = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id, timeout=60).get_data_frames()[0]
print(box.columns.tolist())
box[["familyName", "minutes", "points", "comment"]].head(8)

['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'teamSlug', 'personId', 'firstName', 'familyName', 'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage', 'threePointersMade', 'threePointersAttempted', 'threePointersPercentage', 'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage', 'reboundsOffensive', 'reboundsDefensive', 'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers', 'foulsPersonal', 'points', 'plusMinusPoints']


,familyName,minutes,points,comment
0,Johnson,35:15,23,
1,Williams,34:38,14,
2,Sharpe,30:33,9,
3,Johnson,30:36,21,
4,Russell,27:13,11,
5,Wilson,24:31,15,
6,Martin,22:37,5,
7,Beekman,18:37,0,
